In [1]:

# ============================================================
# 1. IMPORT LIBRARIES + REPRODUCIBILITY SETTINGS
# ============================================================

import os
import random
import time
import subprocess
import warnings
warnings.filterwarnings("ignore")

# Fixed seeds make the model selection stable between different runs.
# This prevents the best model from changing randomly every time the notebook runs.
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

import pandas as pd
import numpy as np
np.random.seed(RANDOM_STATE)

import joblib
import matplotlib.pyplot as plt

from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")
print("XGBoost available:", XGBOOST_AVAILABLE)
print("Random seed fixed at:", RANDOM_STATE)


Libraries loaded successfully.
XGBoost available: True
Random seed fixed at: 42


In [2]:
import kagglehub
import os

dataset_path = kagglehub.dataset_download(
    "quackquackrp/international-graduates-employment-dataset"
)

csv_file = os.path.join(dataset_path, "dataset.csv")
df = pd.read_csv(csv_file)

100%|██████████| 4.01M/4.01M [00:00<00:00, 101MB/s]

Extracting files...


In [3]:

# ============================================================
# 3. DATA UNDERSTANDING
# ============================================================

print("Dataset Information:")
print(df.info())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nMissing Values:")
display(df.isnull().sum())

print("\nEmployment Status Distribution:")
display(df["Employment_Status"].value_counts())

print("\nDescriptive Statistics:")
display(df.describe(include="all").T)


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Country_of_Origin       300000 non-null  object 
 1   Education_Level         300000 non-null  object 
 2   Field_of_Study          300000 non-null  object 
 3   Language_Proficiency    300000 non-null  object 
 4   Visa_Type               300000 non-null  object 
 5   Gender                  300000 non-null  object 
 6   University_Ranking      300000 non-null  object 
 7   Region_of_Study         300000 non-null  object 
 8   Age                     300000 non-null  int64  
 9   Years_Since_Graduation  300000 non-null  int64  
 10  GPA                     300000 non-null  float64
 11  Internship_Experience   300000 non-null  object 
 12  Employment_Status       300000 non-null  object 
 13  Salary                  300000 non-null  int64  
 14 

,0
Country_of_Origin,0
Education_Level,0
Field_of_Study,0
Language_Proficiency,0
Visa_Type,0
Gender,0
University_Ranking,0
Region_of_Study,0
Age,0
Years_Since_Graduation,0



Employment Status Distribution:


,count
Employment_Status,
Employed,156644
Continuing Education,74617
Unemployed,68739



Descriptive Statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Country_of_Origin,300000,8,Brazil,37751,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Education_Level,300000,4,Bachelor's,120143,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Field_of_Study,300000,6,Arts,50144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Language_Proficiency,300000,4,Intermediate,90133,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Visa_Type,300000,4,Student,119754,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,300000,3,Male,143890,NaN,NaN,NaN,NaN,NaN,NaN,NaN
University_Ranking,300000,3,Medium,149847,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region_of_Study,300000,4,Australia,75233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,300000.0,NaN,NaN,NaN,30.49595,5.187951,22.0,26.0,30.0,35.0,39.0
Years_Since_Graduation,300000.0,NaN,NaN,NaN,4.50415,2.874557,0.0,2.0,5.0,7.0,9.0
